# Create Arizona Biomedical Research Commission (ABRC) Awards

Creates awards for the **Arizona Biomedical Research Commission** (OpenAlex F4320332140) -- founded
1994 as the Arizona Disease Control Research Commission, renamed ABRC in 2005, placed under the
Arizona Department of Health Services (ADHS) in 2011 and, since 2015, the *Arizona Biomedical
Research Centre* inside ADHS (same acronym), funded from tobacco-tax and lottery revenue.

**Source:** ADHS's ABRC site, *Current Grant Awards*
(https://www.azdhs.gov/biomedical/index.php#grant-awards): one page per Request for Grant
Applications round, each listing every award with institution, principal investigator and title,
by programme. `scripts/local/abrc_to_s3.py` renders the public page in a headless browser
(Playwright; the section HTML is loaded by script) and reads the tables.

| round | awarded | New Investigator Awards (NIA, up to $250,000 / 3 yrs) | Arizona Investigator Grants (IG, up to $750,000 / 3 yrs) |
|---|---|---|---|
| RFGA2022-010 | January 2023 | 17 | 8 |
| RFGA2023-008 | April 2024 | 20 | 7 |
| RFGA2024-022 | March 2025 | 23 | 8 |

**83 awards, 2023-2025, 100% title / PI / institution.**

**Which years are public:** only these three rounds are published as award lists. FY2007-FY2016
awards appear only inside narrative annual-report PDFs (per-grant pages with PI, institution,
annual amount and end date, no award numbers; 1985-2006 reports are a zip); they are not ingested
here (follow-up candidate).

**amount:** NULL. The pages give only the programme ceiling, which stays in the raw table
(`programme_ceiling_usd`); §6.7 amount check waived because no per-award amount is published.
**start_date:** the first day of the award month the site states ("Awarded March 2025" ->
2025-03-01); `end_date` NULL.

**`funder_award_id` (§2.1.1):** papers cite `RFGA2023-008-11`-style codes (round + application
sequence), which the site does not print. 16 awards carry the code via a crosswalk resolved from the
56 OpenAlex works citing an RFGA code (the round's PI who authors >=50% of the citing works, no tie,
one code per PI; `award_code_source = 'citation_crosswalk'`); the other 67 ship the stable synthetic
key `ABRC-{RFGA}-{pi-slug}`.

**Twins:** Arizona Department of Health Services F4320309677 shares ABRC's ROR (01smpj292) and
holds 388 priority-0 citation rows (16 RFGA-form); F4320332140 holds 492 (38 RFGA-form). Awards
ship under F4320332140 (the tracker's id).

**Prerequisites:** run `scripts/local/abrc_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/abrc/abrc_projects.parquet` (§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320332140`, Arizona Biomedical Research
Commission, US; ror 01smpj292 and doi 10.13039/100008335 read from `openalex.funders.funders`.

**Priority:** `575` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.abrc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/abrc/abrc_projects.parquet`;

In [ ]:
%sql
SELECT rfga, programme, award_code_source, COUNT(*) as n
FROM openalex.awards.abrc_raw
GROUP BY 1, 2, 3 ORDER BY 1, 2, 3;

In [ ]:
%sql
DESCRIBE openalex.awards.abrc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.abrc_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'ABRC F4320332140 must have exactly one openalex.funders.funders row') as funder_ok,
       first(display_name) as display_name, first(ror_id) as ror_id, first(doi) as doi, first(country_code) as country_code
FROM openalex.funders.funders
WHERE funder_id = 4320332140;

## Step 2: Create ABRC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.abrc_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320332140  -- Arizona Biomedical Research Commission
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'abrc_grant_awards' as provenance,
    TRY_TO_DATE(g.award_date, 'yyyy-MM-dd') as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.award_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    named_struct(
        'given_name', NULLIF(TRIM(g.pi_given_name), ''),
        'family_name', NULLIF(TRIM(g.pi_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'US',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    array(named_struct(
        'given_name', NULLIF(TRIM(g.pi_given_name), ''),
        'family_name', NULLIF(TRIM(g.pi_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'US',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    )) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.abrc_raw g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check (fails the run BEFORE the insert): crosswalked rows carry the cited
-- RFGAyyyy-nnn-NN(N) code, all others the ABRC-RFGAyyyy-nnn-<pi-slug> key; ids unique; no rows lost.
SELECT
    assert_true(COUNT_IF(NOT (funder_award_id RLIKE '^RFGA[0-9]{4}-[0-9]{3}-[0-9]{2,3}$'
                              OR funder_award_id RLIKE '^ABRC-RFGA[0-9]{4}-[0-9]{3}-[a-z0-9-]+$')) = 0,
                'funder_award_id is not an RFGA code or ABRC synthetic key') as shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') as ids_unique,
    assert_true(COUNT(*) = (SELECT COUNT(*) FROM openalex.awards.abrc_raw), 'rows dropped in transform') as all_rows,
    COUNT(*) as total
FROM openalex.awards.abrc_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'abrc_grant_awards' AND priority = 575;

-- Insert into openalex_awards_raw with priority.
-- Priority 575: direct-from-funder ingest of ABRC's grant-award pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    575 as priority
FROM openalex.awards.abrc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.abrc_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.abrc_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.abrc_awards
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- §6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.abrc_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.abrc_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 coverage (expect 100% title/start_date/PI/institution; amount NULL by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.abrc_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs (same funder): crosswalked RFGA codes upgrade these.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(DISTINCT c.id) as distinct_awards
FROM openalex.awards.openalex_awards_raw c
JOIN openalex.awards.abrc_awards t ON t.id = c.id
WHERE c.funder_id = 4320332140 AND c.priority = 0
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'abrc_grant_awards'
GROUP BY provenance, priority;